In [14]:

import numpy as np
from sentence_transformers import SentenceTransformer

In [15]:
encoding = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7609.35it/s]


In [16]:
import json
import feedparser #it is a library that parses RSS feeds and returns a Python object that can be easily manipulated.
with open("feeds.json") as f:
    feeds = json.load(f)["feeds"]


for url in feeds:
    feed = feedparser.parse(url)

    for article in feed.entries:
        print(article.title)
        print(article.link)
        print(article.published)
        print("-" * 40)

UK government expected to announce restrictions on trade with West Bank settlements
https://www.bbc.co.uk/news/articles/c4grdjnle22o?at_medium=RSS&at_campaign=rss
Tue, 08 Sep 2026 01:50:55 GMT
----------------------------------------
A&E did not get the basics right - now my son's life is ruined at 32
https://www.bbc.co.uk/news/articles/cqlw0ke2v97o?at_medium=RSS&at_campaign=rss
Tue, 08 Sep 2026 03:49:39 GMT
----------------------------------------
Minister condemns disorder at Portsmouth anti-migrant protest after police officers hurt
https://www.bbc.co.uk/news/articles/cqxv2335je1o?at_medium=RSS&at_campaign=rss
Mon, 07 Sep 2026 19:04:09 GMT
----------------------------------------
Prince George set for first day at Eton College
https://www.bbc.co.uk/news/articles/cwyzel3qjvzo?at_medium=RSS&at_campaign=rss
Tue, 08 Sep 2026 01:00:32 GMT
----------------------------------------
Canadian counter-tariffs come into force escalating US trade dispute
https://www.bbc.co.uk/news/articles/c8jde

In [17]:
import sqlite3

conn = sqlite3.connect("news.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    url TEXT UNIQUE,
    source TEXT,
    published TEXT,
    content TEXT,
    embedded INTEGER DEFAULT 0
)
""")

conn.commit()



In [18]:
cursor.execute("""
SELECT id, url
FROM articles
WHERE content IS NULL
""")
articles= cursor.fetchall() 

from newspaper import Article

for article_id, url in articles:

    art = Article(url)
    art.download()
    art.parse()

    content = art.text

    cursor.execute("""
    UPDATE articles
    SET content = ?
    WHERE id = ?
    """, (content, article_id))

conn.commit()



In [19]:

## chunking the text into smaller pieces for better processing and embedding

from langchain_text_splitters import RecursiveCharacterTextSplitter

def splitter(text, chunk_size=1000, chunk_overlap=200):
    split = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = split.split_text(text)

    ##print(f"Created {len(chunks)} chunks")

    return chunks

In [20]:
from logging import exception
import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print("Initializing embedding model...")
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
        except Exception as e:
            print(f"Error loading model: {e}")

    def Get_embadding(self, texts) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        embedding = self.model.encode(texts, show_progress_bar=True)
        return embedding


    

In [21]:
import time
import chromadb
from newspaper import Article


# =========================================================
# INITIALIZE ONCE
# =========================================================

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="my_collection_new1"
)

embedding_manager = EmbeddingManager()

EMBED_BATCH_LIMIT = 20  # process at most this many articles per cycle, avoids long first-run blocking


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def get_unembedded_articles(cursor, limit=EMBED_BATCH_LIMIT):
    cursor.execute("""
        SELECT id, content, source, title
        FROM articles
        WHERE embedded = 0
        AND content IS NOT NULL
        LIMIT ?
    """, (limit,))
    return cursor.fetchall()


def mark_embedded(conn, article_id):
    conn.execute(
        "UPDATE articles SET embedded = 1 WHERE id = ?",
        (article_id,)
    )
    conn.commit()


# =========================================================
# MAIN PIPELINE
# =========================================================

def after_10minutes():

    total_start = time.time()

    print("\n" + "=" * 50)
    print("STARTING NEWS INGESTION")
    print("=" * 50)


    # -----------------------------------------------------
    # 1. RSS INGESTION
    # -----------------------------------------------------

    rss_start = time.time()

    print("\n[1] Fetching RSS feeds...")

    new_articles = 0

    for feed_url in feeds:

        try:
            feed = feedparser.parse(feed_url)
            source_name = feed.feed.get("title", feed_url)  # FIX: capture outlet name from the feed

            for article in feed.entries:

                title = article.get("title", "")
                url = article.get("link", "")
                published = article.get("published", "")

                if not url:
                    continue

                cursor.execute("""
                    INSERT OR IGNORE INTO articles
                    (title, url, source, published)
                    VALUES (?, ?, ?, ?)
                """, (
                    title,
                    url,
                    source_name,   # FIX: source now actually stored
                    published
                ))

                if cursor.rowcount > 0:
                    new_articles += 1

        except Exception as e:
            print(f"RSS feed failed: {feed_url}")
            print(f"Error: {e}")

    conn.commit()

    print(f"New articles found: {new_articles}")
    print(
        f"RSS time: {time.time() - rss_start:.2f} seconds"
    )


    # -----------------------------------------------------
    # 2. DOWNLOAD ARTICLE CONTENT
    # -----------------------------------------------------

    download_start = time.time()

    print("\n[2] Downloading article content...")

    cursor.execute("""
        SELECT id, url
        FROM articles
        WHERE content IS NULL
        LIMIT ?
    """, (EMBED_BATCH_LIMIT * 2,))  # cap this stage too, so downloads don't run away on first launch

    articles_to_download = cursor.fetchall()

    print(
        f"Articles to download: "
        f"{len(articles_to_download)}"
    )

    downloaded = 0

    for article_id, url in articles_to_download:

        try:

            art = Article(url)

            art.download()
            art.parse()

            content = art.text

            if not content:
                print(
                    f"No content: article {article_id}"
                )
                continue

            cursor.execute("""
                UPDATE articles
                SET content = ?
                WHERE id = ?
            """, (
                content,
                article_id
            ))

            downloaded += 1

        except Exception as e:

            print(
                f"Failed article {article_id}: {e}"
            )

    conn.commit()

    print(f"Successfully downloaded: {downloaded}")

    print(
        f"Download time: "
        f"{time.time() - download_start:.2f} seconds"
    )


    # -----------------------------------------------------
    # 3. GET UNEMBEDDED ARTICLES
    # -----------------------------------------------------

    embedding_start = time.time()

    print("\n[3] Finding articles to embed...")

    articles = get_unembedded_articles(cursor)

    print(
        f"Articles waiting for embeddings (this cycle, capped at {EMBED_BATCH_LIMIT}): "
        f"{len(articles)}"
    )


    # -----------------------------------------------------
    # 4. CHUNK + EMBED + STORE
    # -----------------------------------------------------

    total_chunks = 0
    embedded_articles = 0

    for article_id, content, source, title in articles:

        try:

            # DIAGNOSTIC: catch abnormally large content (bad scrape) before chunking
            print(
                f"Article {article_id}: content length = {len(content)} chars"
            )
            if len(content) > 20000:
                print(
                    f"  WARNING: unusually long content, likely a bad scrape — skipping"
                )
                mark_embedded(conn, article_id)  # mark done so it doesn't retry forever
                continue

            # Chunk
            chunks = splitter(content)

            if not chunks:
                print(
                    f"No chunks: article {article_id}"
                )
                continue

            print(
                f"Article {article_id}: "
                f"{len(chunks)} chunks"
            )

            total_chunks += len(chunks)


            # Embeddings (progress bar off — adds overhead in a loop)
            embeddings = (
                embedding_manager.Get_embadding(chunks)
            )
            if hasattr(embeddings, "tolist"):
                embeddings = embeddings.tolist()


            # IDs
            ids = [
                f"{article_id}_{i}"
                for i in range(len(chunks))
            ]


            # Metadata
            metadatas = [
                {
                    "article_id": int(article_id),
                    "source": str(source or ""),
                    "title": str(title or "")
                }
                for _ in chunks
            ]


            # Store in ChromaDB
            collection.add(
                embeddings=embeddings,
                documents=chunks,
                metadatas=metadatas,
                ids=ids
            )


            # Only mark after successful Chroma insertion
            mark_embedded(
                conn,
                article_id
            )

            embedded_articles += 1

        except Exception as e:

            print(
                f"Embedding failed for article "
                f"{article_id}: {e}"
            )


    # -----------------------------------------------------
    # 5. SUMMARY
    # -----------------------------------------------------

    embedding_time = time.time() - embedding_start
    total_time = time.time() - total_start

    print("\n" + "=" * 50)
    print("INGESTION COMPLETE")
    print("=" * 50)

    print(
        f"Articles embedded: {embedded_articles}"
    )

    print(
        f"Total chunks: {total_chunks}"
    )

    print(
        f"Embedding time: {embedding_time:.2f} seconds"
    )

    print(
        f"TOTAL TIME: {total_time:.2f} seconds"
    )

    print("=" * 50)

import schedule
import time

# Run immediately
after_10minutes()

# Then every 10 minutes
schedule.every(10).minutes.do(after_10minutes)

while True:
    schedule.run_pending()
    time.sleep(30)

Initializing embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13426.99it/s]



STARTING NEWS INGESTION

[1] Fetching RSS feeds...
New articles found: 35
RSS time: 0.78 seconds

[2] Downloading article content...
Articles to download: 35
Successfully downloaded: 35
Download time: 17.70 seconds

[3] Finding articles to embed...
Articles waiting for embeddings (this cycle, capped at 20): 20
Article 1: content length = 3048 chars
Article 1: 4 chunks


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.57it/s]


Article 2: content length = 1298 chars
Article 2: 2 chunks


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 31.40it/s]


Article 37: content length = 1213 chars
Article 37: 2 chunks


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.06it/s]



INGESTION COMPLETE
Articles embedded: 15
Total chunks: 35
Embedding time: 0.67 seconds
TOTAL TIME: 1.18 seconds


KeyboardInterrupt: 

In [22]:
print(collection.count())

200


In [23]:
data = collection.get(
    include=["documents", "embeddings", "metadatas"]
)

In [24]:
print(data["documents"][0])
print(data["embeddings"][0])
print(data["metadatas"][0])

Cleveland Police's Chief Constable has said she is proud of her "brave" force but that it was being "let down" by the way it was funded, following a fatal crash which killed seven people including two police officers.

In her first sit-down interview since the A66 crash, speaking exclusively to the BBC, Victoria Fuller said the force needed to become part of a larger force with greater resources.

She said her officers are dealing with some of the highest crime levels in the country akin to those faced by metropolitan forces, but with less than half the number of officers.

Multiple people have since been arrested as part of a wider police operation against organised crime following the crash.

A Volkswagen Passat carrying five young men drove the wrong way down the motorway near Middlesbrough in the early hours of the morning before colliding with a marked police car.
[ 4.67160195e-02  8.26066080e-03  1.66078899e-02  3.89323421e-02
  6.56207949e-02  7.96061456e-02 -1.99663546e-02  6.1

# this where i am strat adding tools and llm

In [25]:
import os 
from dotenv import load_dotenv
load_dotenv

from langchain_tavily import TavilySearch

from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from  typing_extensions import TypedDict
from typing import Annotated,Optional
from langgraph.graph.message import add_messages

from langchain.chat_models import init_chat_model

model = init_chat_model(
    "groq:llama-3.3-70b-versatile"
)


embedding_model = Embediingmanager() 

class State(TypedDict):
    messages: Annotated[list, add_messages]
    query: str
    retrieved_docs: list[dict]           # chunks from retrieval
    route_decision: Optional[str]        # "single_search" or "compare_outlets"
    outlet_results: Optional[dict]       # for cross-outlet comparison
    answer: Optional[str]






def retrieve_node(state: State):
    query = state["query"]
    query_embedding = embedding_model.Get_embadding(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=5)
    
    docs = []
    for text, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        docs.append({"text": text, "metadata": meta, "distance": dist})
    
    return {"retrieved_docs": docs}

def is_retrieval_sufficient(state: State) -> str:
    best_distance = state["retrieved_docs"][0]["distance"]
    return "fallback" if best_distance > 0.35 else "generate"

def generate_node(state: State):
    query = state["query"]
    docs = state["retrieved_docs"]
    
    context = "\n\n".join(
        f"[{d['metadata'].get('source', 'unknown')}] {d['text']}" 
        for d in docs
    )
    
    prompt = f"""Answer the question using only the context below. 
If the context doesn't contain enough information, say so.

Context:
{context}

Question: {query}

Answer:"""
    
    response = model.invoke(prompt)
    
    return {
        "answer": response.content,
        "messages": [response]
    }


def fallback_node(state: State):
    query = state["query"]
    
    tavily = TavilySearch(max_results=3)
    results = tavily.invoke({"query": query})
    
    # normalize Tavily's output to the same shape as your Chroma retrieved_docs
    # so generate_node doesn't need to know which source it came from
    docs = []
    for r in results.get("results", []):
        docs.append({
            "text": r.get("content", ""),
            "metadata": {
                "source": r.get("url", "web"),
                "title": r.get("title", "")
            },
            "distance": None  # not applicable for web search results
        })
    
    return {"retrieved_docs": docs}






def tool_calling (query):
    return {"messages" :model.invoke(query)}

grapghbuilder = StateGraph(State) 


grapghbuilder.add_node("retreiver",retrieve_node)
grapghbuilder.add_node("generate_answer",generate_node)
grapghbuilder.add_node("web_search",fallback_node)




grapghbuilder.add_edge(START,"retreiver")
grapghbuilder.add_conditional_edges(
    "retreiver",
    is_retrieval_sufficient,
    {
        "generate": "generate_answer",
        "fallback": "web_search"
    }
)

grapghbuilder.add_edge("web_search", "generate_answer")
grapghbuilder.add_edge("generate_answer", END)

app = grapghbuilder.compile()




GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
result = app.invoke({"query": "latest news about india ", "messages": []})
print(result["answer"])

In [ ]:
import schedul
import time 

def get_updated_news():
    
